In [1]:
import glob, json, os, time
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import f1_score, recall_score, precision_score

MODELS = "../models"
DATA = "../data/SKAB/data"
cfg = json.load(open(f"{MODELS}/config.json"))
WINDOW, FEATURES, THR = cfg["window"], cfg["features"], cfg["ae_threshold"]
scaler = joblib.load(f"{MODELS}/scaler.pkl")
ae = tf.keras.models.load_model(f"{MODELS}/autoencoder.keras")

def load(p):
    return pd.read_csv(p, sep=";", index_col="datetime", parse_dates=True)

Xs, ys = [], []
for f in sorted(glob.glob(f"{DATA}/valve1/*.csv") + glob.glob(f"{DATA}/valve2/*.csv")):
    df = load(f)
    X = scaler.transform(df[FEATURES].values)
    y = df["anomaly"].values
    for i in range(0, len(X) - WINDOW + 1, 1):
        Xs.append(X[i:i + WINDOW].flatten())
        ys.append(int(y[i:i + WINDOW].mean() > 0.5))
X_test = np.array(Xs, dtype=np.float32)
y_test = np.array(ys)
print(X_test.shape)

(21852, 256)


In [2]:
conv = tf.lite.TFLiteConverter.from_keras_model(ae)
open(f"{MODELS}/ae_float32.tflite", "wb").write(conv.convert())

conv = tf.lite.TFLiteConverter.from_keras_model(ae)
conv.optimizations = [tf.lite.Optimize.DEFAULT]       # dynamic-range quantization
open(f"{MODELS}/ae_quant.tflite", "wb").write(conv.convert())

for n in ["autoencoder.keras", "ae_float32.tflite", "ae_quant.tflite"]:
    print(f"{n}: {os.path.getsize(f'{MODELS}/{n}') / 1024:.1f} KB")

INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmpsiwblbmy\assets


INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmpsiwblbmy\assets


Saved artifact at 'C:\Users\ASUS\AppData\Local\Temp\tmpsiwblbmy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 256), dtype=tf.float32, name=None)
Captures:
  2909965542864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965543248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965544592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965543824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965544016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965543632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965545360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965542480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965545936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965544784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965546320

INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmpvgn6ro4g\assets


Saved artifact at 'C:\Users\ASUS\AppData\Local\Temp\tmpvgn6ro4g'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 256), dtype=tf.float32, name=None)
Captures:
  2909965542864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965543248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965544592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965543824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965544016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965543632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965545360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965542480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965545936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965544784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2909965546320

In [3]:
def run_tflite(path, X):
    interp = tf.lite.Interpreter(model_path=path, num_threads=1)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]["index"]
    out = interp.get_output_details()[0]["index"]
    errs, times = [], []
    for x in X:
        x = x.reshape(1, -1)
        t = time.perf_counter()
        interp.set_tensor(inp, x)
        interp.invoke()
        y = interp.get_tensor(out)
        times.append((time.perf_counter() - t) * 1000)
        errs.append(np.mean((x - y) ** 2))
    return np.array(errs), np.array(times)

rows = []
for name in ["ae_float32", "ae_quant"]:
    errs, times = run_tflite(f"{MODELS}/{name}.tflite", X_test)
    pred = (errs > THR).astype(int)
    rows.append({
        "Model": name,
        "Size (KB)": round(os.path.getsize(f"{MODELS}/{name}.tflite") / 1024, 1),
        "Mean latency (ms)": round(times.mean(), 3),
        "P95 latency (ms)": round(np.percentile(times, 95), 3),
        "Precision": round(precision_score(y_test, pred, zero_division=0), 3),
        "Recall": round(recall_score(y_test, pred), 3),
        "F1": round(f1_score(y_test, pred), 3),
    })
pd.DataFrame(rows)

c:\Users\ASUS\edge-ai-anomaly\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
c:\Users\ASUS\edge-ai-anomaly\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,Model,Size (KB),Mean latency (ms),P95 latency (ms),Precision,Recall,F1
0,ae_float32,333.8,0.010,0.015,0.357,1.0,0.526
1,ae_quant,95.6,0.005,0.008,0.357,1.0,0.526


In [4]:
pd.DataFrame(rows).to_csv(f"{MODELS}/tflite_comparison.csv", index=False)